# Late to Office Prediction

Cleaned version of the original notebook. The project uses Logistic Regression as the primary model and Decision Tree for comparison.


In [ ]:
!pip install pandas numpy matplotlib scikit-learn joblib


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


In [ ]:
df = pd.read_csv("late_to_office_dataset.csv")
df.head()


In [ ]:
print("Shape:", df.shape)
df.info()
print("\nMissing values:\n", df.isnull().sum())
display(df.describe())


In [ ]:
plt.figure(figsize=(8, 6))
colors = df["will_be_late"].map({0: "green", 1: "red"})
plt.scatter(df["distance_km"], df["time_left_minutes"], c=colors, alpha=0.6)
plt.xlabel("Distance from Home (km)")
plt.ylabel("Time Left Before Late (minutes)")
plt.title("Will You Be Late?")

legend_elements = [
    Patch(facecolor="green", label="On Time (0)"),
    Patch(facecolor="red", label="Late (1)")
]
plt.legend(handles=legend_elements)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

X = df[["distance_km", "time_left_minutes"]]
y = df["will_be_late"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)

lr_predictions = lr.predict(X_test_scaled)
print("Logistic Regression accuracy:", accuracy_score(y_test, lr_predictions))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_predictions))


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)

dt_predictions = dt.predict(X_test_scaled)
print("Decision Tree accuracy:", accuracy_score(y_test, dt_predictions))
print("Confusion matrix:\n", confusion_matrix(y_test, dt_predictions))


In [ ]:
# Decision boundary for the Logistic Regression model
x_min, x_max = df["distance_km"].min() - 1, df["distance_km"].max() + 1
y_min, y_max = df["time_left_minutes"].min() - 1, df["time_left_minutes"].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = pd.DataFrame({
    "distance_km": xx.ravel(),
    "time_left_minutes": yy.ravel()
})

grid_scaled = scaler.transform(grid)

# Correct: the trained Logistic Regression model is named lr
Z = lr.predict(grid_scaled).reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlGn_r")
plt.scatter(
    df["distance_km"],
    df["time_left_minutes"],
    c=colors,
    alpha=0.6,
    edgecolors="k",
    linewidths=0.3
)
plt.xlabel("Distance from Home (km)")
plt.ylabel("Time Left Before Late (minutes)")
plt.title("Decision Boundary - Will You Be Late?")
plt.legend(handles=legend_elements)
plt.show()


In [ ]:
import joblib

joblib.dump(lr, "late_to_office_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved successfully.")


In [ ]:
# Example prediction
sample = pd.DataFrame({
    "distance_km": [15],
    "time_left_minutes": [20]
})

sample_scaled = scaler.transform(sample)
prediction = lr.predict(sample_scaled)
probability = lr.predict_proba(sample_scaled)

print("Prediction:", "Late" if prediction[0] == 1 else "On Time")
print(
    "Probability - On Time: {:.2f}, Late: {:.2f}".format(
        probability[0][0], probability[0][1]
    )
)
